# 03 - TextRank Phase 1 Benchmark

This notebook is the official Phase 1 benchmark artifact view for the TextRank extractive baseline.

Objectives:
- document the TextRank setup under the frozen Phase 0 protocol
- report official TextRank benchmark outputs on validation subset
- compare TextRank against TF-IDF on the same subset and config
- summarize key failure cases and practical trade-offs

## 1. Protocol And Method

Phase-0 alignment:
- Protocol version: `phase0_v2`
- Split: `validation`
- Candidate `top_k`: `[2, 3, 4, 5]`
- Subset limit: `200`
- Article threshold: `article_char_len >= 1200`

TextRank method used in backend:
- Similarity strategy: cosine similarity over sentence TF vectors (`cosine-tf`)
- Sentence graph: weighted, undirected graph from pairwise sentence similarity
- Ranking: PageRank over the sentence graph
- Selection modes: `top_k` (preferred) or `ratio` (fallback when top_k is not provided)

In [8]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 220)

ROOT = Path.cwd().resolve()
if (ROOT / "backend").exists():
    PROJECT_ROOT = ROOT
elif (ROOT.parent / "backend").exists():
    PROJECT_ROOT = ROOT.parent
else:
    raise RuntimeError("Repository root not found. Open notebook from repo root or notebooks/.")

OFFICIAL_DIR = PROJECT_ROOT / "notebooks" / "results" / "official" / "validation"
if not OFFICIAL_DIR.exists():
    raise FileNotFoundError(f"Official directory not found: {OFFICIAL_DIR}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("OFFICIAL_DIR:", OFFICIAL_DIR)

PROJECT_ROOT: C:\Users\OS\OneDrive\Desktop\Text_Summarization
OFFICIAL_DIR: C:\Users\OS\OneDrive\Desktop\Text_Summarization\notebooks\results\official\validation


In [9]:
def latest_file(pattern: str) -> Path:
    files = sorted(OFFICIAL_DIR.glob(pattern))
    if not files:
        raise FileNotFoundError(f"No files match pattern: {pattern}")
    return files[-1]


def extract_ts(path: Path, prefix: str) -> str:
    name = path.name
    start = len(prefix)
    end = name.rfind(".")
    return name[start:end]


textrank_report_path = latest_file("textrank_phase1_topk_report_*.json")
textrank_report = json.loads(textrank_report_path.read_text(encoding="utf-8"))
textrank_ts = extract_ts(textrank_report_path, "textrank_phase1_topk_report_")

textrank_summary_path = Path(textrank_report["summary_csv"])
textrank_detail_path = Path(textrank_report["detail_csv"])
textrank_error_path = OFFICIAL_DIR / f"textrank_phase1_error_analysis_{textrank_ts}.md"
if not textrank_error_path.exists():
    raise FileNotFoundError(
        f"Expected TextRank error analysis for the same timestamp ({textrank_ts}) not found: {textrank_error_path}"
    )

compare_report_path = OFFICIAL_DIR / f"engine_compare_report_{textrank_ts}.json"
if not compare_report_path.exists():
    raise FileNotFoundError(
        f"Expected engine comparison report for TextRank timestamp ({textrank_ts}) not found: {compare_report_path}"
    )
compare_report = json.loads(compare_report_path.read_text(encoding="utf-8"))
compare_summary_path = Path(compare_report["summary_csv"])

paths_snapshot = {
    "textrank_timestamp": textrank_ts,
    "textrank_report": str(textrank_report_path),
    "textrank_summary": str(textrank_summary_path),
    "textrank_detail": str(textrank_detail_path),
    "textrank_error_analysis": str(textrank_error_path),
    "engine_compare_report": str(compare_report_path),
    "engine_compare_summary": str(compare_summary_path),
}
paths_snapshot

{'textrank_timestamp': '20260426_000926',
 'textrank_report': 'C:\\Users\\OS\\OneDrive\\Desktop\\Text_Summarization\\notebooks\\results\\official\\validation\\textrank_phase1_topk_report_20260426_000926.json',
 'textrank_summary': 'C:\\Users\\OS\\OneDrive\\Desktop\\Text_Summarization\\notebooks\\results\\official\\validation\\textrank_phase1_topk_summary_20260426_000926.csv',
 'textrank_detail': 'C:\\Users\\OS\\OneDrive\\Desktop\\Text_Summarization\\notebooks\\results\\official\\validation\\textrank_phase1_topk_detail_20260426_000926.csv',
 'textrank_error_analysis': 'C:\\Users\\OS\\OneDrive\\Desktop\\Text_Summarization\\notebooks\\results\\official\\validation\\textrank_phase1_error_analysis_20260426_000926.md',
 'engine_compare_report': 'C:\\Users\\OS\\OneDrive\\Desktop\\Text_Summarization\\notebooks\\results\\official\\validation\\engine_compare_report_20260426_000926.json',
 'engine_compare_summary': 'C:\\Users\\OS\\OneDrive\\Desktop\\Text_Summarization\\notebooks\\results\\officia

## 2. Official TextRank Results

In [10]:
textrank_report = json.loads(textrank_report_path.read_text(encoding="utf-8"))
textrank_summary_df = pd.read_csv(textrank_summary_path)

official_snapshot = {
    "protocol_version": textrank_report.get("protocol_version"),
    "target_split": textrank_report.get("target_split"),
    "top_k_candidates": textrank_report.get("top_k_candidates"),
    "subset_limit": textrank_report.get("subset_limit"),
    "article_char_threshold": textrank_report.get("article_char_threshold"),
    "recommended_top_k_by_weighted_rank": textrank_report.get("recommended_top_k_by_weighted_rank"),
    "official_top_k": textrank_report.get("official_top_k"),
    "official_top_k_rationale": textrank_report.get("official_top_k_rationale"),
}

official_snapshot

{'protocol_version': 'phase0_v2',
 'target_split': 'validation',
 'top_k_candidates': [2, 3, 4, 5],
 'subset_limit': 200,
 'article_char_threshold': 1200,
 'recommended_top_k_by_weighted_rank': 2,
 'official_top_k': 2,
 'official_top_k_rationale': 'Locked to weighted-rank winner using the same protocol as TF-IDF (ROUGE-1/2/L, compression_ratio, repetition_rate, latency).'}

In [11]:
textrank_summary_df

,engine,top_k,n,rouge1_f,rouge2_f,rougeL_f,summarizer_core_latency_sec,compression_ratio,repetition_rate
0,textrank,2,200,0.498134,0.223585,0.320491,0.003679,0.139735,0.098919
1,textrank,3,200,0.431449,0.211267,0.290172,0.003560,0.206669,0.102362
2,textrank,4,200,0.365660,0.193336,0.256340,0.003408,0.278047,0.105334
3,textrank,5,200,0.319969,0.181851,0.233435,0.003578,0.343811,0.110362


## 3. TF-IDF vs TextRank Comparison

In [12]:
compare_report = json.loads(compare_report_path.read_text(encoding="utf-8"))
compare_summary_df = pd.read_csv(compare_summary_path)

compare_report.get("best_by_engine", {})

{'tfidf': {'top_k': 2,
  'rouge1_f': 0.4773284952590099,
  'rouge2_f': 0.14002458274886284,
  'rougeL_f': 0.2701445692164762,
  'compression_ratio': 0.1162247958155047,
  'repetition_rate': 0.010818618758739136,
  'summarizer_core_latency_sec': 0.00048507349885767325,
  'weighted_rank_score': 1.6500000000000001},
 'textrank': {'top_k': 2,
  'rouge1_f': 0.49813430188724916,
  'rouge2_f': 0.22358461216448394,
  'rougeL_f': 0.32049108667498594,
  'compression_ratio': 0.13973500856940563,
  'repetition_rate': 0.0989193219892886,
  'summarizer_core_latency_sec': 0.003678693001129432,
  'weighted_rank_score': 1.1500000000000001}}

In [13]:
pivot_df = compare_summary_df.pivot(index="top_k", columns="engine", values=["rouge1_f", "rouge2_f", "rougeL_f", "summarizer_core_latency_sec", "compression_ratio", "repetition_rate"])
pivot_df

rouge1_f            rouge2_f            rougeL_f            \
engine  textrank     tfidf  textrank     tfidf  textrank     tfidf   
top_k                                                                
2       0.498134  0.477328  0.223585  0.140025  0.320491  0.270145   
3       0.431449  0.451618  0.211267  0.156097  0.290172  0.259615   
4       0.365660  0.401438  0.193336  0.166818  0.256340  0.243885   
5       0.319969  0.349952  0.181851  0.165413  0.233435  0.224916   

       summarizer_core_latency_sec           compression_ratio            \
engine                    textrank     tfidf          textrank     tfidf   
top_k                                                                      
2                         0.003679  0.000485          0.139735  0.116225   
3                         0.003560  0.000566          0.206669  0.182509   
4                         0.003408  0.000496          0.278047  0.245340   
5                         0.003578  0.000484          0.343811  0.311420   

       repetition_rate            
engine        textrank     tfidf  
top_k                             
2             0.098919  0.010819  
3             0.102362  0.015170  
4             0.105334  0.020908  
5             0.110362  0.024708

## 4. Short Failure Cases

Below are short failure cases exported from the official TextRank error analysis artifact.

In [14]:
error_text = textrank_error_path.read_text(encoding="utf-8")
print("Error analysis file:", textrank_error_path)
print("\n".join(error_text.splitlines()[:80]))

Error analysis file: C:\Users\OS\OneDrive\Desktop\Text_Summarization\notebooks\results\official\validation\textrank_phase1_error_analysis_20260426_000926.md
# TEXTRANK Short Error Analysis

- Official `top_k`: `2`
- Sample size: `200`
- Mean ROUGE-1/2/L: `0.4981` / `0.2236` / `0.3205`
- Mean compression ratio: `0.1397`
- Mean repetition rate: `0.0989`

## Failure Cases (3 lowest ROUGE-L)
### Case 1 - guid 18826
- ROUGE-1/2/L: `0.1854` / `0.0733` / `0.1391`
- Article snippet: Buổi giao_lưu giữa dịch_giả Nguyễn_Bá_Quỳnh với bạn_đọc tại Đường sách TP. HCM sáng 18-1 nhằm giới_thiệu quyển sách Nhấn nút tái_tạo - tác_giả Satya_Nadella , CEO của Microsoft - là cuộc chuyện_trò mang nhiều gợi_mở cho cái nhìn về công_nghệ ứng_dụng trong tương_lai . Sách được chú_ý không_chỉ bởi nó được một trong 3 CEO lừng_danh của Microsoft viết ra , tất_nhiên , vai_trò của Satya trong việc định_hướng cho Microsoft để có_thể chuyển_đổi cục_diện của tập_đoàn này khi lần_lượt các đối_thủ đang 
- Reference summary

## 5. Conclusion

- TextRank achieves higher ROUGE than TF-IDF on the same validation subset and configuration.
- TextRank is slower than TF-IDF in summarizer-core latency.
- TextRank has higher repetition rate than TF-IDF, so repetition must be monitored in deployment/reporting.

Recommended reporting statement:
- Keep `top_k=2` as the official TextRank setting for this run (weighted-rank winner).
- Present TextRank as quality-strong baseline, with explicit caveat on speed and repetition trade-offs.